# Spatial perturbation benchmark — the phase-1 pipeline

Evaluates how well a model predicts the **spatial effect of a CRISPR perturbation**: when a gene is knocked out, how does the perturbed cell itself change, and how does that change spread to its spatial neighbours? The same cell is never measured both perturbed and unperturbed, so everything is scored at the **distribution level** against a **control reference**. Run on the lab server (env `spatial-tumor-ai`), where the Saunders `.h5mu` slices and the offline scGEN dumps live.

## 0. What's new in this refactor (read this first)

**1. Two-layer adapters.** An *input* adapter (`SaundersAdapter`) normalises a raw `.h5mu` slice into one `StandardData`. An *output* adapter (`models.scgen_model.ScgenSeedModel`) normalises every model's native output. Models never touch each other's formats.

**2. Sample-level aggregate control** (`reference_aggregate.aggregate_control` -> `harness._control_reference_aggregate`). The reference 'unperturbed' state is now a per-cell-type aggregate over CONTROL cells only — no nearest-neighbour matched-control pairing in feature space, so no matched-control leakage. The SAME aggregate-control reference now drives EVERY comparison: the seed baseline (`seed_ref`) and the niche no-effect baseline (`e_null` = the average same-type control cell's neighbourhood) both come from `reference_aggregate.control_reference_centers`. `run_benchmark` uses this by default.

**3. `eval_X` — one unified scoring space.** PCC-delta is not cross-space robust, so the prediction, the observed cells, and the delta baseline must all be compared in the SAME space. For scGEN the natural space is its log-norm training space: we pass `eval_X=lognorm_X` (the `build_lognorm_X` matrix) into `fill_2x2`, which slices the observed/reference seed cells into that same matrix so scGEN's seed is scored fairly.

**4. MC-spatial quadrant x dimension stratification.** MC-spatial flags, per guide, whether a real **Self** (X) and/or **Niche** (Y) effect exists, giving four quadrants: **Both**, **X-Only (Self)**, **Y-Only (Niche)**, **Inert**. We score **seed only where there is Self signal** (X-Only/Both), **niche only where there is Niche signal** (Y-Only/Both), and use the **Inert** quadrant as a rigorous **negative control** (a good model should predict ~no effect there). This avoids diluting a niche score over perturbations that have no niche signal.

## 1. Config
All server-specific paths live here, at the top, so a server run only edits this one cell. `SLICE_DIR` is the Saunders directory; `MAX_FILES` how many slices to pool. The two OPTIONAL inputs — `SCGEN_DUMP_DIR` (offline `{P}_seed.h5ad` dumps) and `DUAL_METRICS_CSV` (MC-spatial `*_Dual_Metrics.csv`) — may be left as `None`/missing; the notebook then runs baseline-only / unstratified and says so explicitly.

In [ ]:
from pathlib import Path

# --- required: the Saunders slice(s) ---
SLICE_DIR = '/home/yiru/database/spatial_perturbed_processed/CRISPR_based/Saunders_2025_40513557'
MAX_FILES = 1                      # pool this many .h5mu slices (1 = a single slice)
COUNTS_LAYER = 'X'                 # raw-count layer for scGEN log-norm export (mod/rna/X)

# --- optional input A: offline scGEN seed dumps ({P}_seed.h5ad), one per perturbation ---
SCGEN_DUMP_DIR = None             # e.g. '/home/yiru/scgen_dumps/Saunders_slice0'; None -> skip scGEN

# --- optional input B: MC-spatial four-quadrant labels for this slice ---
DUAL_METRICS_CSV = None           # e.g. '/home/yiru/mc_spatial_out/slice0_Dual_Metrics.csv'; None -> unstratified

# --- evaluation perturbation list (MC-spatial Self/Niche-significant guides for this slice) ---
EVAL = ['Pck1','Rrbp1','Hspd1','Psmc1','Sepp1','Bcl2l1','Vcp',
        'Ass1','Pten','Rrn3','Letm1','Hspa5','Sec61b','Rngtt']

K_GRAPH, K_REF, NICHE_HOPS = 15, 5, 1
print('SLICE_DIR        =', SLICE_DIR)
print('SCGEN_DUMP_DIR   =', SCGEN_DUMP_DIR, '(scGEN seed will run only if these dumps exist)')
print('DUAL_METRICS_CSV =', DUAL_METRICS_CSV, '(quadrant stratification only if present)')

## 1b. Imports

In [ ]:
import matplotlib
%matplotlib inline
import numpy as np, pandas as pd

from spbench.adapters.saunders import SaundersAdapter
from spbench.adapters.scgen_export import build_lognorm_X
from spbench.graph import build_knn_graph
from spbench.niche import build_spatial_graph, compute_niche_composition
from spbench.reference_aggregate import aggregate_control
from spbench.config import run_benchmark
from spbench.harness import (fill_2x2, _control_reference_aggregate, _control_residuals)
from spbench.compare import evaluate_seed
from spbench.models.trivial_seed import TrivialSeed
from spbench.models.gaussian_prop import GaussianProp
from spbench.models.gcn_prop import SimpleGCN
from spbench import mc_spatial_join as mcj
from spbench import mc_spatial_report as mcr

## 2. Load the data
`SaundersAdapter(SLICE_DIR, max_files=MAX_FILES, counts_layer='X').load()` reads the raw `.h5mu` slice(s) and normalises them into one `StandardData`: `data.X` is the (z-scored) `layers/raw_scaled` matrix; `counts_layer='X'` ALSO loads the raw integer counts into `data.meta['counts']` (needed for scGEN's log-norm space in §6).

In [ ]:
data = SaundersAdapter(SLICE_DIR, max_files=MAX_FILES, counts_layer=COUNTS_LAYER).load()
n_pert = len(data.perturbations())
print(f'{data.n_cells} cells, {data.n_genes} genes, {n_pert} perturbations, '
      f'{int(data.is_control.sum())} control cells, {int(data.is_perturbed.sum())} perturbed cells')

# Keep only EVAL perturbations actually present in this slice (defensive).
present = set(data.perturbations())
EVAL = [p for p in EVAL if p in present]
print('evaluating', len(EVAL), 'perturbations present in this slice:', EVAL)
assert EVAL, 'none of the EVAL perturbations are present in this slice — check SLICE_DIR / EVAL'

## 3. Niche graph + cell-type composition (composition input)
Two interchangeable spatial graphs, both returning a `(2, n_edges)` `[src, dst]` array built WITHIN each slice (no cross-batch edges, no self-loops):
- `build_knn_graph(data, k)` — pure sklearn kNN (the harness/`run_benchmark` default).
- `build_spatial_graph(data, n_neighs)` — squidpy-backed (optional; needs squidpy installed).

`compute_niche_composition(data, edges, hops)` then gives a `(n_cells, C)` row-simplex of each cell's neighbourhood cell-type composition — the **composition** quantity. We build the kNN graph here and reuse it everywhere below so the niche definition is consistent.

In [ ]:
edges = build_knn_graph(data, k=K_GRAPH)
print('kNN graph:', edges.shape[1], 'directed edges')

# Optional squidpy graph (same (2, n_edges) format); falls back to the kNN graph if squidpy
# is not installed in this env.
try:
    sq_edges = build_spatial_graph(data, n_neighs=K_GRAPH)
    print('squidpy spatial graph:', sq_edges.shape[1], 'edges (drop-in alternative to kNN)')
except Exception as e:
    print('squidpy graph unavailable (', type(e).__name__, '), using kNN graph only')

comp = compute_niche_composition(data, edges, hops=NICHE_HOPS)   # (n_cells, C) composition row-simplex
C = comp.shape[1]
print('niche composition (composition):', comp.shape, '->', C, 'cell types; row sums ~',
      float(np.nanmean(comp.sum(1)[comp.sum(1) > 0])))

## 4. MC-spatial quadrants (optional)
MC-spatial writes a `*_Dual_Metrics.csv` per slice with two axis p-values per guide: `p_val_specificity` (X = a real **Self/seed** effect) and `p_val_global` (Y = a real **Niche/niche** effect). `mc_spatial_join.load_dual_metrics(csv, p_cutoff, p_mode)` reads it (stdlib `csv`, no pandas) and tags each perturbation with its quadrant — `BOTH`, `X_ONLY`, `Y_ONLY`, `INERT` (label constants live in `mc_spatial_join`). If `DUAL_METRICS_CSV` is unset or missing, we fall back to the unstratified EVAL list and clearly note that MC-spatial was not provided.

In [ ]:
QUADRANT_OF = {}        # perturbation -> quadrant label (empty => no MC-spatial labels)
HAVE_MC = bool(DUAL_METRICS_CSV) and Path(DUAL_METRICS_CSV).exists()
if HAVE_MC:
    dual = mcj.load_dual_metrics(DUAL_METRICS_CSV, p_cutoff=0.05, p_mode='raw')
    QUADRANT_OF = {d['perturbation']: d['quadrant'] for d in dual}
    from collections import Counter
    counts = Counter(QUADRANT_OF.get(p, mcj.UNKNOWN) for p in EVAL)
    print('MC-spatial quadrants for the EVAL guides:')
    for q in (mcj.BOTH, mcj.X_ONLY, mcj.Y_ONLY, mcj.INERT, mcj.UNKNOWN):
        if counts.get(q):
            print(f'  {q:18s}: {counts[q]}')
    print('  (Inert is the negative control; seed is scored on Self/Both, niche on Niche/Both.)')
else:
    print('NOTE: MC-spatial Dual_Metrics.csv not provided (DUAL_METRICS_CSV is unset/missing).')
    print('      -> running UNSTRATIFIED on the EVAL list; the §7 stratified report is skipped.')

## 5. Run the baselines (seed + niche)
`run_benchmark` now uses the **sample-level aggregate control** by default (`_control_reference_aggregate`) — no matched-control feature pairing. For each perturbation it composes a **TrivialSeed** (seed floor), a **Gaussian-kernel** baseline propagation and a self-supervised **GCN** learned propagation (both niche), and scores them.

**We report everything as `E-distance`: LOWER = better, with NO subtraction.** Energy distance is always **>= 0**, so we report the (mean) energy distance to the observed perturbed cells directly — no `log` wrapper (the niche E is small, well under 1, because it is a neighbourhood aggregate, but it is never negative). From `res['seed'][p]['e_samples']` and `res['compare'][p]['e']`:
- **seed** (`seed_E`) — the seed/TrivialSeed E; `seed_null_E` is the **control-seed reference** (the same-type aggregate-control seed).
- **niche** (`niche_E`) — the deployable `model+learned` niche E; `niche_null_E` is the **no-effect reference** (the aggregate same-type control niche — the neighbourhood of the average unperturbed cell of that type), and `niche_oracle_E` is the **leakage-free ceiling**.

`null` and `oracle` are **reference values shown alongside, never subtracted** — a method beats no-effect when its `*_E` sits **below** `*_null_E`. PCC-delta (`seed_pcc`/`niche_pcc`) is kept only as a *supplementary* column.

In [ ]:
res = run_benchmark(data, perturbations=EVAL, k=K_GRAPH, k_ref=K_REF,
                    gcn_kwargs={'hidden': 64, 'epochs': 30})

# Everything below is reported as E-distance: LOWER = better, NO subtraction. Energy
# distance is always >= 0, so we report the raw (mean) E with no log wrapper.
# null (no-effect aggregate-control niche) and oracle (leakage-free ceiling) are
# REFERENCE values shown alongside, never subtracted. PCC-delta is supplementary.
def _meanE(v):
    """Mean per-repeat E (linear, >=0); nan when there are no samples."""
    return float(np.nanmean(v)) if np.size(v) else float('nan')

rows = []
for p in EVAL:
    s = res['seed'][p]; c = res['compare'][p]
    es = s.get('e_samples', {})
    rows.append(dict(
        perturbation=p,
        quadrant=QUADRANT_OF.get(p, mcj.UNKNOWN),
        seed_E=_meanE(es.get('model', [])),                 # seed (TrivialSeed) E
        seed_null_E=_meanE(es.get('null', [])),             # seed control-seed reference
        niche_E=float(c['e']['model+learned']),             # niche (deployable) E
        niche_null_E=float(c['e']['null']),                 # niche no-effect reference
        niche_oracle_E=(float(c['e']['oracle']) if 'oracle' in c['e'] else float('nan')),  # niche ceiling ref
        seed_pcc=s['pcc_delta'], niche_pcc=c['pcc']['model+learned'],  # supplementary (PCC-delta)
        leak_ok=res['leakage_pass'][p],
    ))
# Sort by niche E ascending (lower = better); null/oracle columns are references, not scores.
base_df = pd.DataFrame(rows).sort_values('niche_E', ascending=True).reset_index(drop=True)
base_df

## 5b. Seed / niche method-comparison box plots (the headline view)
Mirrors the CONCERT-style figure but as **three boards**: **one box per method**, box = that method's pooled **per-repeat matched-n energy distance** (`res[...]['e_samples']`), shown as **E-distance (linear, >=0), lower = better**.
- **seed** — all models together; dashed `null (floor)` = same-type control center. **No upper bound** (GT seed = the observed center itself -> energy ~0, trivial).
- **niche . seed-only + Gaussian** — box = model seed propagated by the shared Gaussian baseline; upper bound = **GT seed + Gaussian** (cell-1, the baseline-prop ceiling, isolates seed quality).
- **niche . end-to-end (GCN mock)** — box = the learned-prop niche; upper bound = **GT seed + GCN** (cell-2). GCN stands in as a mock end-to-end model until a real one (CONCERT) is wired in.
Upper bounds are **per-prop** (each prop has its own GT-seed ceiling). This is `plot_seed_prop(res)`.

In [ ]:
from spbench.plotting import plot_seed_prop, collect_seed_samples, collect_niche_tier
fig = plot_seed_prop(res)
fig.savefig('seed_prop_methods.png', dpi=130, bbox_inches='tight')
_sb, _ = collect_seed_samples(res)
_b1, _d1 = collect_niche_tier(res, 'base'); _b2, _d2 = collect_niche_tier(res, 'learned')
print('seed board   :', list(_sb), '(floor only, no upper bound)')
print('niche tier-1 :', list(_b1), '| refs', list(_d1), '(Gaussian, upper = GT+Gaussian)')
print('niche tier-2 :', list(_b2), '| refs', list(_d2), '(GCN mock, upper = GT+GCN)')
fig

## 5c. PCC-delta — the unified primary metric (seed | niche)
Seed and niche on ONE logic: **PCC-delta of the perturbation shift** (predicted Delta vs observed Delta, baseline = matched control), **higher = better, bounded**. This is mean-shift accuracy — the quantity that is actually comparable for models predicting a conditional mean. The E-distance box in 5b is the variance-aligned **secondary** view: seed now gets the same control-residual restoration as niche, so a collapsed mean-field seed is no longer structurally penalised by the energy distance (that was a metric artefact, not a model failure). Dashed: perfect=1, no-corr=0. This is `plot_delta(res)`.

In [ ]:
from spbench.plotting import plot_delta, collect_delta
figd = plot_delta(res)
figd.savefig('delta_methods.png', dpi=130, bbox_inches='tight')
_ds, _ = collect_delta(res, 'seed'); _dn, _ = collect_delta(res, 'niche')
print('seed  PCC-delta methods:', list(_ds))
print('niche PCC-delta methods:', list(_dn))
figd

## 6. scGEN — a seed-only conditional model (optional)
scGEN is run OFFLINE in its own env; per perturbation it dumps `{P}_seed.h5ad` (a per-center-aligned predicted-seed array). `ScgenSeedModel({P: path})` loads it and serves it as the model seed; `.centers(P)` gives the StandardData center indices the rows align to. We score scGEN's **seed** in its own log-norm space: build `lognorm_X = build_lognorm_X(data)` (raw counts -> normalize_total 1e4 + log1p) and pass `eval_X=lognorm_X` into `fill_2x2`. `fill_2x2` then slices the observed/reference seed cells into that same matrix (the three are co-spaced) and carries `eval_X=None` downstream, so `evaluate_seed(niches)` scores fairly.

If `SCGEN_DUMP_DIR` has no `{P}_seed.h5ad` for any EVAL perturbation, this cell is a clearly marked no-op and the notebook stays baseline-only.

In [ ]:
from spbench.models.scgen_model import ScgenSeedModel

scgen_paths = {}
if SCGEN_DUMP_DIR:
    for p in EVAL:
        f = Path(SCGEN_DUMP_DIR) / f'{p}_seed.h5ad'
        if f.exists():
            scgen_paths[p] = str(f)

scgen_seed = {}        # perturbation -> {'pcc_delta','mse','n'} for scGEN's seed
if scgen_paths:
    print(f'scGEN dumps found for {len(scgen_paths)}/{len(EVAL)} perturbations.')
    # scGEN's seed_pred lives in the log-norm space -> score there via eval_X=lognorm_X.
    lognorm_X = build_lognorm_X(data)                  # (n_cells, n_genes) log-norm matrix
    X_ref_agg = _control_reference_aggregate(data, edges)   # aggregate-control reference (G1)
    resid = _control_residuals(data)
    base_prop = GaussianProp().fit(data, edges)
    learned_prop = SimpleGCN(hidden=64, epochs=30).fit(data, edges)
    for p, path in scgen_paths.items():
        seed_model = ScgenSeedModel({p: path})         # offline loader; predict_seed is 2-arg
        grid = fill_2x2(data, p, edges, seed_model, base_prop, learned_prop, k_ref=K_REF,
                        X_ref=X_ref_agg, return_niches=True, residuals=resid,
                        eval_X=lognorm_X)              # log-norm scoring space for seed
        niches = grid['_niches']                        # eval_X already consumed into the matrix
        scgen_seed[p] = evaluate_seed(niches, eval_X=niches.get('eval_X'))   # carried eval_X is None here
    # scGEN seed as E-distance (linear, >=0; lower = better); pcc kept supplementary.
    sc_df = pd.DataFrame([dict(perturbation=p, quadrant=QUADRANT_OF.get(p, mcj.UNKNOWN),
                               scgen_seed_E=_meanE(v.get('e_samples', {}).get('model', [])),
                               scgen_seed_pcc=v['pcc_delta'])
                          for p, v in scgen_seed.items()])
    base_df = base_df.merge(sc_df.drop(columns='quadrant'), on='perturbation', how='left')
    display(sc_df.sort_values('scgen_seed_E', ascending=True).reset_index(drop=True))
else:
    print('NOTE: no scGEN {P}_seed.h5ad dumps found (SCGEN_DUMP_DIR unset/empty).')
    print('      -> running BASELINE-ONLY; scGEN seed columns are omitted.')

### Combined per-perturbation results table
One row per perturbation, all in **E-distance (linear, >=0), lower = better**: the baselines' **seed** (`seed_E`, with `seed_null_E` reference) and **niche** (`niche_E`, with `niche_null_E` and `niche_oracle_E` references), plus scGEN's seed (`scgen_seed_E`) when its dumps were found. PCC-delta columns (`seed_pcc`/`niche_pcc`/`scgen_seed_pcc`) are supplementary. This is the per-perturbation evidence behind the capability matrix in §7.

In [ ]:
base_df

## 7. Capability matrix + stratified report
The **capability matrix** below reports each model x dimension as **mean E-distance (linear, >=0; LOWER = better)**, with `null`/`oracle` as **reference rows** (never subtracted). A model is only scored on the dims it covers (NaN otherwise).

The optional **stratified report** then joins the MC-spatial quadrant onto each model and reports, **per dimension, only over the quadrants where that dimension has signal**, with **Inert as the negative control**:
- **seed** (self) — scored on **X-Only + Both** (`mc_spatial_report.DIMENSION_QUADRANTS['d1']`).
- **niche** — scored on **Y-Only + Both** (`DIMENSION_QUADRANTS['d2']`).
- **Inert** — negative control: a good model has `mean_gain ~ 0` there.

`mc_spatial_report.stratified_report(records, dim_gain_fields)` consumes records that carry a `quadrant` label (from `join_quadrants`) plus per-dimension gain fields. Here those gain fields are an **internal** quantity derived from the E columns — how far each method sits **below** its no-effect `null` E (positive = better) — used only to drive the quadrant stratification and the Inert negative-control read; the headline tables above remain E-distance. If MC-spatial was not provided, we show the unstratified capability matrix above only.

In [ ]:
def _capability_matrix(df):
    """Mean E-distance (linear, >=0) per model x dimension (LOWER = better); null/oracle are
    reference rows, not models. NaN = dimension not covered by that model."""
    cm = {'TrivialSeed (seed)': {'seed': float(np.nanmean(df['seed_E'])), 'niche': np.nan, 'composition': np.nan},
          'Gaussian/GCN (niche)': {'seed': np.nan,
                                   'niche': float(np.nanmean(df['niche_E'])), 'composition': np.nan},
          'null (reference)': {'seed': float(np.nanmean(df['seed_null_E'])),
                               'niche': float(np.nanmean(df['niche_null_E'])), 'composition': np.nan},
          'oracle (reference)': {'seed': np.nan,
                                 'niche': float(np.nanmean(df['niche_oracle_E'])), 'composition': np.nan},
          'niche-composition module': {'seed': np.nan, 'niche': np.nan, 'composition': 'simplex (built in §3)'}}
    if 'scgen_seed_E' in df.columns:
        cm['scGEN (seed)'] = {'seed': float(np.nanmean(df['scgen_seed_E'])), 'niche': np.nan, 'composition': np.nan}
    return pd.DataFrame(cm).T

print('Capability matrix — mean E-distance (linear, >=0), lower = better; null/oracle = reference rows')
display(_capability_matrix(base_df))

In [ ]:
if HAVE_MC:
    # The headline tables above are E-distance (linear); this stratified report still uses an
    # INTERNAL gain (how far a method sits BELOW its no-effect null E, so positive = better)
    # purely to drive the MC-spatial quadrant stratification / negative-control read.
    records = []
    for _, r in base_df.iterrows():
        # seed gain = how far the seed E sits below the control-seed reference (null) E.
        seed_ref = r['seed_null_E']
        seed_self = r['scgen_seed_E'] if ('scgen_seed_E' in base_df.columns
                                          and pd.notna(r.get('scgen_seed_E'))) else r['seed_E']
        seed_gain = float(seed_ref - seed_self)
        # niche gain = how far the niche E sits below the no-effect null E.
        niche_gain = float(r['niche_null_E'] - r['niche_E'])
        records.append({'perturbation': r['perturbation'],
                        'gain_d1': seed_gain, 'gain_d2': niche_gain})
    # Attach quadrants from the CSV (left join; absent guides -> 'Unknown').
    records = mcj.join_quadrants(records, DUAL_METRICS_CSV, key='perturbation', p_mode='raw')
    report = mcr.stratified_report(records, dim_gain_fields={'d1': 'gain_d1', 'd2': 'gain_d2'})
    rep_df = pd.DataFrame(report)
    print('Stratified report (seed on Self/Both, niche on Niche/Both, Inert = negative control):')
    display(rep_df)
    print('Read: mean_gain should be > 0 in the signal rows and ~ 0 in the is_negative_control'
          ' (Inert) rows.')
else:
    print('MC-spatial not provided -> stratified report skipped. Unstratified capability'
          ' matrix above is the summary; rerun with DUAL_METRICS_CSV set for the quadrant'
          ' x dimension breakdown with the Inert negative control.')

## 8. Bottom line — how to read it
Everything is **E-distance (linear, >=0): LOWER = better**, with `null`/`oracle` shown as references (never subtracted).
- **Capability matrix.** Each model is scored only on the dims it covers: scGEN on **seed**, the Gaussian/GCN propagation on **niche**, the niche module supplies **composition**. NaN means 'dim not covered', not 'failed'. `null` and `oracle` are reference rows.
- **scGEN vs TrivialSeed on seed.** scGEN is a conditional model; its seed E should sit **below TrivialSeed's** (lower = closer to the observed seed) and **near the `null` reference on Inert** in the stratified read. A seed far below null on Inert would mean scGEN is hallucinating an effect where MC-spatial found none.
- **niche.** `model+learned` E **below `niche_null_E`** beats predicting 'the neighbours did not change', and that advantage should be concentrated in the **Niche/Both** quadrants, not in Inert; `niche_oracle_E` is the leakage-free floor.
- **eval_X.** scGEN's seed is scored in its log-norm space; the baselines' seed is in `data.X` space — compare each model to its own `null` reference, not the raw E numbers across spaces.

In [ ]:
# All numbers are E-distance (linear, >=0), LOWER = better; null/oracle are references (never subtracted).
has_scgen = 'scgen_seed_E' in base_df.columns
print('Per-dimension coverage this run:')
print('  seed (self)         : TrivialSeed' + (', scGEN' if has_scgen else '')
      + '  (scored on Self/Both quadrants)')
print('  niche (niche expr)  : Gaussian + GCN (model+learned)  (scored on Niche/Both quadrants)')
print('  composition         : compute_niche_composition simplex, shape', comp.shape)
niche_mean = float(np.nanmean(base_df['niche_E'])); niche_null = float(np.nanmean(base_df['niche_null_E']))
n_niche_win = int((base_df['niche_E'] < base_df['niche_null_E']).sum())
print(f'\nniche: mean E {niche_mean:.3f}  vs  null {niche_null:.3f}  (lower = better)')
print(f'    {n_niche_win}/{len(base_df)} guides beat no-effect (niche E below null).')
if has_scgen:
    sc_mean = float(np.nanmean(base_df['scgen_seed_E'])); tr_mean = float(np.nanmean(base_df['seed_E']))
    better = base_df['scgen_seed_E'] < base_df['seed_E']
    print(f'seed: scGEN mean E {sc_mean:.3f}  vs  TrivialSeed {tr_mean:.3f}  (lower = better);'
          f' scGEN lower on {int(better.sum())}/{int(better.notna().sum())} guides.')
if not HAVE_MC:
    print('\n(MC-spatial labels not provided -> the Inert negative control was not evaluated.)')

## 9. Visual summary (the headline result)
A single **capability-matrix heatmap** over **mean E-distance (linear, >=0; LOWER = better)** — each model scored only on the dims it covers ('—' = not covered), with `null`/`oracle` as reference rows. Seed E (~5.5) and niche E (~0.25) live on different scales (~20x apart) and are NOT comparable across dimensions, so each column is **colored independently** (within-column rank, 0 = best) while the annotation is the raw linear E. The per-method seed and niche E box plots are the **§5b** figure (`plot_seed_prop`); this panel is the compact model x dimension summary. The figure is saved next to the notebook as `run_benchmark_results.png`, followed by a clean text dump of the final table.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Single capability-matrix heatmap over mean E-distance (linear, >=0), LOWER = better.
# Seed E (~5.5) and niche E (~0.25) differ ~20x and are NOT comparable across dimensions, so
# a single shared colorbar is meaningless in linear space -> color each column independently
# (within-column rank; 0 = lowest E in that column = best) while annotating the raw linear E.
# (The per-method seed/niche E box plots are the §5b plot_seed_prop figure.)
cm = _capability_matrix(base_df)
num = cm[['seed', 'niche']].apply(pd.to_numeric, errors='coerce').astype(float)
# per-column min-max normalize for COLOR only (cross-dim E magnitudes aren't comparable);
# 0 = lowest E in that column = best
norm = num.copy()
for col in norm.columns:
    v = norm[col].to_numpy(dtype=float)
    finite = np.isfinite(v)
    if finite.sum() > 1:
        lo, hi = np.nanmin(v[finite]), np.nanmax(v[finite])
        norm[col] = (v - lo) / (hi - lo) if hi > lo else 0.5
fig, ax = plt.subplots(figsize=(6, 3.2))
im = ax.imshow(norm.to_numpy(dtype=float), cmap='RdYlBu', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(num.columns))); ax.set_xticklabels(num.columns)
ax.set_yticks(range(len(num.index))); ax.set_yticklabels(num.index, fontsize=7.5)
for i in range(num.shape[0]):
    for j in range(num.shape[1]):
        val = num.to_numpy(dtype=float)[i, j]
        ax.text(j, i, '—' if not np.isfinite(val) else f'{val:.2f}',
                ha='center', va='center', fontsize=9)
ax.set_title('Capability matrix — mean E-distance (per-column color; lower = better)')
fig.colorbar(im, ax=ax, fraction=.046, pad=.04, label='within-column rank (0 = best)')
fig.tight_layout()
fig.savefig('run_benchmark_results.png', dpi=130, bbox_inches='tight')
plt.show()
print('saved figure -> notebooks/run_benchmark_results.png  (see §5b for per-method box plots)')

In [ ]:
# ---- clean text dump of the final results (all E-distance, linear, >=0; lower = better) ----
pd.set_option('display.float_format', lambda v: f'{v:.4f}')
has_scgen = 'scgen_seed_E' in base_df.columns
cols = ['perturbation', 'quadrant', 'seed_E']
if has_scgen:
    cols += ['scgen_seed_E']
cols += ['niche_E', 'niche_null_E', 'niche_oracle_E']
print('=' * 72)
print('FINAL RESULTS  -  Saunders Batch_10_Slice_0  (one row per guide; E-distance, lower = better)')
print('=' * 72)
print(base_df[cols].to_string(index=False))
print('\nCapability matrix (mean E-distance, linear, >=0, lower = better; - = dim not covered; '
      'null/oracle = reference rows):')
print(_capability_matrix(base_df).to_string())
if has_scgen:
    sc_mean = float(np.nanmean(base_df['scgen_seed_E'])); tr_mean = float(np.nanmean(base_df['seed_E']))
    n_win = int((base_df['scgen_seed_E'] < base_df['seed_E']).sum())
    print(f'\nseed: scGEN mean E {sc_mean:.3f}  vs  TrivialSeed mean E {tr_mean:.3f}'
          f'   (lower wins; scGEN lower on {n_win}/{len(base_df)} guides)')
n_niche = int((base_df['niche_E'] < base_df['niche_null_E']).sum())
niche_mean = float(np.nanmean(base_df['niche_E'])); niche_null = float(np.nanmean(base_df['niche_null_E']))
print(f'niche: mean E {niche_mean:.3f}  vs  null {niche_null:.3f}  '
      f'(below null on {n_niche}/{len(base_df)} guides)')